# OR-Bench Over-Refusal Analysis

Measures whether CC-Delta and CAA steering cause **over-refusal** on
benign-but-sensitive prompts using the [OR-Bench](https://arxiv.org/abs/2405.20947)
benchmark. The key metric is **OR-Bench compliance**: the fraction of
benign prompts the model answers (higher is better).

This analysis complements the safety vs utility tradeoff in
`results_analysis.ipynb` by checking a different failure mode:
a safe model that refuses everything is useless.

## How to run experiments with OR-Bench

Add `"orbench"` to the evaluation list in your config, e.g.:

```yaml
evaluation: ["text-generation", "strongreject_finetuned", "strongreject_rubric", "fluency", "orbench"]
```

Then run experiments as usual:

```bash
uv run python -m ccdelta.scripts.run_experiments
```

In [ ]:
from pathlib import Path

import altair as alt
import pandas as pd

from notebooks.notebook_utils import (
    add_safety_score,
    expand_generation_params,
    find_optimal_configs,
    group_mean_scores,
    load_sweep_results,
    plot_multi_metric,
)

In [ ]:
RESULTS_ROOT = Path("./artifacts")  # Change to "./results" for default run_experiments output

BASELINE_DIR = RESULTS_ROOT / "baseline"
SAE_SWEEP_ROOT = RESULTS_ROOT / "sae"
DENSE_SWEEP_ROOT = RESULTS_ROOT / "dense"

In [ ]:
# --- Load experiments ---

dfs = []
for path, label in [(BASELINE_DIR, "baseline"), (SAE_SWEEP_ROOT, "sae"), (DENSE_SWEEP_ROOT, "dense")]:
    if not path.exists():
        print(f"WARNING: {path} not found, skipping {label}")
        continue
    try:
        df = load_sweep_results(path, method_label=label)
        dfs.append(df)
        print(f"Loaded {label}: {len(df)} rows")
    except FileNotFoundError as e:
        print(f"WARNING: {e}")

if not dfs:
    raise RuntimeError("No experiment results found.")

all_results = pd.concat(dfs, ignore_index=True)

if "orbench_compliance" not in all_results.columns:
    raise RuntimeError(
        "OR-Bench results not found. Add 'orbench' to the evaluation list in your config and re-run."
    )

print(f"\nTotal: {len(all_results)} rows, orbench_compliance coverage: {all_results['orbench_compliance'].notna().mean():.0%}")

In [ ]:
# --- Compute metrics ---

all_results = expand_generation_params(all_results)
all_results["safety"] = add_safety_score(all_results)

# Unify CAA strength -> steering_multiplier
if "strength" in all_results.columns:
    all_results["steering_multiplier"] = all_results["steering_multiplier"].fillna(
        all_results["strength"]
    )

In [ ]:
# --- Aggregate per configuration ---

group_cols = ["intervention_type", "generation_param"]
if "model" in all_results.columns:
    group_cols.insert(0, "model")

score_cols = ["safety", "orbench_compliance"]

agg = group_mean_scores(all_results, group_cols=group_cols, score_cols=score_cols)
agg = expand_generation_params(agg)

if "strength" in agg.columns:
    agg["steering_multiplier"] = agg["steering_multiplier"].fillna(agg["strength"])

print(f"Aggregated to {len(agg)} configurations")
agg

In [ ]:
# --- Safety vs OR-Bench Compliance scatter ---

tooltip_cols = [c for c in ["intervention_type", "steering_multiplier",
                             "num_features", "safety", "orbench_compliance",
                             "generation_param"]
                if c in agg.columns]

scatter = alt.Chart(agg).mark_circle(size=100).encode(
    x=alt.X("safety:Q", title="Safety"),
    y=alt.Y("orbench_compliance:Q", title="OR-Bench Compliance",
            scale=alt.Scale(zero=False)),
    color=alt.Color("intervention_type:N", title="Method"),
    tooltip=tooltip_cols,
).properties(
    title="Safety vs OR-Bench Compliance (higher is better)",
    width=500, height=350,
)

if "steering_multiplier" in agg.columns:
    scatter = scatter.encode(
        size=alt.Size("steering_multiplier:Q", title="Steering Multiplier",
                      scale=alt.Scale(range=[40, 300])),
    )

display(scatter)

In [ ]:
# --- Per-method metric breakdown ---

metrics = ["safety", "orbench_compliance"]
baseline_df = agg[agg["intervention_type"] == "baseline"]

sae_df = agg[agg["intervention_type"] == "sae"]
if len(sae_df) > 0 and "steering_multiplier" in sae_df.columns:
    display(plot_multi_metric(
        sae_df, metrics, x_col="steering_multiplier",
        reference_df=baseline_df, title="CC-Delta: Safety & OR-Bench vs Steering Multiplier",
    ))

dense_df = agg[agg["intervention_type"] == "dense"]
if len(dense_df) > 0 and "steering_multiplier" in dense_df.columns:
    display(plot_multi_metric(
        dense_df, metrics, x_col="steering_multiplier",
        reference_df=baseline_df, title="CAA: Safety & OR-Bench vs Steering Multiplier",
    ))

In [ ]:
# --- Best configs per threshold (maximize safety, constrain orbench compliance) ---

baseline_df = agg[agg["intervention_type"] == "baseline"].copy()
intervention_df = agg[agg["intervention_type"] != "baseline"].copy()

if len(baseline_df) > 0 and len(intervention_df) > 0:
    optimal = find_optimal_configs(
        intervention_df,
        baseline_df,
        threshold_changes=[0.05, 0.10, 0.25, 1.0],
        target_score="safety",
        constraint_scores=["orbench_compliance"],
    )

    if len(optimal) > 0:
        display_cols = [c for c in ["intervention_type", "generation_param",
                                     "threshold_change", "safety", "safety_diff",
                                     "orbench_compliance", "orbench_compliance_diff",
                                     "sample_count"]
                        if c in optimal.columns]
        display(optimal[display_cols])
    else:
        print("No valid configurations found at any threshold level.")
else:
    print("Need both baseline and intervention results.")

In [ ]:
# --- Method comparison plot (paper-style) ---

if 'optimal' in dir() and len(optimal) > 0:
    baseline_point = baseline_df.iloc[[0]].copy()
    baseline_point["threshold_change"] = 0.0

    plot_cols = [c for c in ["intervention_type", "threshold_change",
                             "safety", "orbench_compliance", "generation_param"]
                if c in optimal.columns]
    for c in plot_cols:
        if c not in baseline_point.columns:
            baseline_point[c] = None
    plot_df = pd.concat([baseline_point[plot_cols], optimal[plot_cols]], ignore_index=True)

    points = alt.Chart(plot_df).mark_circle(size=80).encode(
        x=alt.X("safety:Q", title="Safety"),
        y=alt.Y("orbench_compliance:Q", title="OR-Bench Compliance",
                scale=alt.Scale(zero=False)),
        color=alt.Color("intervention_type:N", title="Method"),
        size=alt.Size("threshold_change:Q", title="Max Compliance Loss",
                      scale=alt.Scale(range=[30, 200])),
        tooltip=[c for c in plot_cols if c in plot_df.columns],
    )

    lines = alt.Chart(plot_df).mark_line(
        strokeDash=[8, 2], opacity=0.7
    ).encode(
        x="safety:Q",
        y="orbench_compliance:Q",
        color="intervention_type:N",
        detail="intervention_type:N",
    )

    chart = (lines + points).properties(
        title="Safety vs OR-Bench Compliance (best configs per threshold)",
        width=500, height=350,
    )
    display(chart)
else:
    print("Run the optimal configs cell above first.")